In [1]:
# --- Cell 2: paths + imports ---
import site
import sys
from pathlib import Path

if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

candidate_cutlass_paths = [
    Path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"),
    Path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages/"),
]
for p in candidate_cutlass_paths:
    if p.exists():
        site.addsitedir(str(p))
        print(f"Added cutlass path: {p}")
        break

import pandas as pd
import torch
import kagglehub
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    import mamba_ssm  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "mamba_ssm import failed. Run cell 1, restart kernel, then re-run from cell 2."
    ) from exc

OUTPUT_DIR = "/kaggle/working"
MODEL_ID = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
LORA_RANK = 32

# Training knobs (reduce for smoke tests)
MAX_TRAIN_SAMPLES = None  # e.g. 256 for a quick run
NUM_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 4096

# Match nvidia-nemotron-metric.ipynb user suffix
BOXED_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

Added cutlass path: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages


/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


torch: 2.12.0.dev20260324+cu128 cuda: True


In [2]:
# --- Cell 3: resolve train.csv (Kaggle competition input or cwd) ---
from pathlib import Path

def resolve_train_csv() -> Path:
    candidates = [
        Path("/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"),
        Path("/kaggle/input/competitions/nvidia-nemotron-3-reasoning-challenge/train.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    found = list(Path("/kaggle/input").rglob("train.csv")) if Path("/kaggle/input").exists() else []
    if found:
        return found[0]
    local = Path("train.csv")
    if local.exists():
        return local.resolve()
    raise FileNotFoundError("train.csv not found. Add the competition dataset as a Kaggle input.")

train_path = resolve_train_csv()
df = pd.read_csv(train_path)
if MAX_TRAIN_SAMPLES is not None:
    df = df.head(int(MAX_TRAIN_SAMPLES)).copy()
print("train_path:", train_path, "rows:", len(df))

train_path: /kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv rows: 9500


In [3]:
# --- Cell 4: tokenizer + chat formatting (mirrors metric + docs/training-data-format.md) ---
import os
from pathlib import Path
MODEL_PATH = None
for path in ["/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default", "/kaggle/input/nemotron-3-nano/transformers/default/1"]:
    if Path(path).exists():
        MODEL_PATH = path
        print(f"Found model in Kaggle inputs: {MODEL_PATH}")
        break
if MODEL_PATH is None:
    print("Downloading model via kagglehub (requires internet)...")
    MODEL_PATH = kagglehub.model_download(MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token


def example_to_text(prompt: str, answer: str) -> str:
    user_content = str(prompt).strip() + BOXED_SUFFIX
    assistant_content = f"\\boxed{{{str(answer).strip()}}}"
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]
    kwargs = dict(tokenize=False, add_generation_prompt=False)
    try:
        return tokenizer.apply_chat_template(messages, **kwargs, enable_thinking=True)
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kwargs)


texts = [example_to_text(row.prompt, row.answer) for row in df.itertuples(index=False)]
train_ds = Dataset.from_dict({"text": texts})
print("sample text length (chars):", len(texts[0]) if texts else 0)

sample text length (chars): 673


In [4]:
# --- Cell 5: model + LoRA ---
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116


In [ ]:
# --- Cell 6: Training (Replaced TRL with transformers.Trainer for offline mode) ---
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Fallback definitions in case they were lost from memory
OUTPUT_DIR = globals().get("OUTPUT_DIR", "/kaggle/working")
NUM_EPOCHS = globals().get("NUM_EPOCHS", 1)
PER_DEVICE_TRAIN_BATCH_SIZE = globals().get("PER_DEVICE_TRAIN_BATCH_SIZE", 1)
GRADIENT_ACCUMULATION_STEPS = globals().get("GRADIENT_ACCUMULATION_STEPS", 8)
LEARNING_RATE = globals().get("LEARNING_RATE", 2e-4)
MAX_SEQ_LENGTH = globals().get("MAX_SEQ_LENGTH", 4096)

training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/sft_checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    report_to="none",
)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_SEQ_LENGTH)

print("Tokenizing dataset...")
tokenized_train_ds = train_ds.map(tokenize_function, batched=True)
tokenized_train_ds = tokenized_train_ds.remove_columns(["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_ds,
    data_collator=data_collator,
)

# Fix Kaggle read-only permission error on Triton binaries
import os
import shutil
import subprocess

ro_bin_dir = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin"
rw_bin_dir = "/kaggle/working/triton_bin"
if os.path.exists(ro_bin_dir):
    os.makedirs(rw_bin_dir, exist_ok=True)
    for f in os.listdir(ro_bin_dir):
        src = os.path.join(ro_bin_dir, f)
        dst = os.path.join(rw_bin_dir, f)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
        os.chmod(dst, 0o777)

    orig_popen = subprocess.Popen
    def patched_popen(*args, **kwargs):
        if args and isinstance(args[0], list) and isinstance(args[0][0], str):
            if args[0][0].startswith(ro_bin_dir):
                args[0][0] = args[0][0].replace(ro_bin_dir, rw_bin_dir)
        return orig_popen(*args, **kwargs)
    subprocess.Popen = patched_popen

trainer.train()

Tokenizing dataset...


Map:   0%|          | 0/9500 [00:00<?, ? examples/s]

Step,Training Loss
10,17.971924
20,8.450672
30,8.760637
40,7.797084
50,7.189707
60,7.418492
70,5.977983
80,7.333167
90,6.667341
100,7.795071


In [1]:
# --- Cell 6 (Export): Save trained adapter weights ---
model.save_pretrained(OUTPUT_DIR)
print("Saved adapter under", OUTPUT_DIR)

NameError: name 'model' is not defined

In [ ]:
# --- Cell 7 (Submit): Zip adapter files for Kaggle submission ---
import subprocess
from pathlib import Path

out = Path(OUTPUT_DIR)
cfg = out / "adapter_config.json"
weights = out / "adapter_model.safetensors"
if not weights.exists():
    weights = out / "adapter_model.bin"
if not cfg.exists() or not weights.exists():
    raise FileNotFoundError(f"Missing adapter files. Expected {cfg} and a weights file.")

subprocess.run(
    f"cd {OUTPUT_DIR} && zip -m submission.zip adapter_config.json {weights.name}",
    shell=True,
    check=True,
)
print("Wrote", out / "submission.zip")